# PISA 2022 Screen Time Data — Analysis with Pandas

**IU000116 Computational Practices: Visualisation & Sensing**  
BSc Creative Computing · UAL CCI · 2026

This notebook documents the data analysis stage of the project. The raw dataset is an Excel file published by the OECD as part of their PISA 2022 Policy Insights report (PIF N°124). 

Unlike a typical CSV, this file has multiple sheets — one per figure/table — and messy headers with many blank rows. A key part of this analysis is understanding and cleaning that structure before doing anything useful with it.

**Dataset:** OECD (2023). *PIF N°124 – Figures & Tables*. PISA 2022 Results – Volume II.  
https://doi.org/10.1787/53f23881-en

**References:**  
- McKinney, W. (2022). *Python for Data Analysis*, 3rd ed. O'Reilly. https://wesmckinney.com/book/  
- Wickham, H. (2014). Tidy Data. *Journal of Statistical Software*, 59(10). https://r4ds.had.co.nz/tidy-data.html  
- CM Hub (2022). *Data processing with Pandas*. https://github.com/Pecnut/course-pandas

---
## 1. Setup — import libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import json
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

print('Libraries loaded.')

---
## 2. Load the Excel file — inspect the sheets

The PISA data comes as a multi-sheet `.xlsx` file, not a single flat CSV. This is common in real-world research data. `pd.read_excel()` can handle this — we first inspect the sheet names, then load each one we need.

This mirrors the pandas course approach to `exam_results.xlsx` — loading from Excel, then exploring the structure before doing analysis.

In [ ]:
FILE = 'PIF_124_Figures_Tables.xlsx'

# Inspect all sheet names
xl = pd.ExcelFile(FILE)
print(f'Total sheets: {len(xl.sheet_names)}')
print('Sheet names:', xl.sheet_names)

---
## 3. Load & clean Figure 1 — Distraction by country

**What this sheet contains:** The percentage of students in each country who report being distracted by their own device, and by classmates' devices, in most maths lessons.

**The cleaning challenge:** The OECD has formatted this sheet for presentation, not analysis. There are ~98 blank/header rows before the actual data starts. This is a classic real-world data wrangling problem — the kind that Wes McKinney dedicates a whole chapter to in *Python for Data Analysis*.

In [ ]:
# Load raw — no header, so we can see the full structure
raw_fig1 = pd.read_excel(FILE, sheet_name='Figure 1', header=None)
print(f'Raw shape: {raw_fig1.shape}')

# Find where the actual data starts (row with column headers)
# The header row contains 'Students get distracted by using digital devices'
header_row = raw_fig1[raw_fig1[0].astype(str).str.contains('Japan', na=False)].index[0] - 1
print(f'Data header at row: {header_row}')

In [ ]:
# Re-load skipping the blank rows, using the correct header
df_distraction = pd.read_excel(
    FILE,
    sheet_name='Figure 1',
    header=None,
    skiprows=header_row + 1  # skip metadata + column header row
)

# Keep only the 3 columns we need: country name + two distraction percentages
df_distraction = df_distraction.iloc[:, :3].copy()
df_distraction.columns = ['country', 'self_distracted_pct', 'others_distracted_pct']

# Drop rows where country is NaN (blank separator rows)
df_distraction = df_distraction.dropna(subset=['country'])

# Drop rows where both percentages are NaN
df_distraction = df_distraction.dropna(subset=['self_distracted_pct', 'others_distracted_pct'])

# Clean country names — strip asterisks (OECD uses * for data caveats)
df_distraction['country'] = df_distraction['country'].str.replace(r'\*', '', regex=True).str.strip()

# Round to 2 decimal places
df_distraction[['self_distracted_pct', 'others_distracted_pct']] = \
    df_distraction[['self_distracted_pct', 'others_distracted_pct']].round(2)

# Reset index
df_distraction = df_distraction.reset_index(drop=True)

print(f'Clean shape: {df_distraction.shape}')
df_distraction.head(10)

### 3a. Basic inspection — following the pandas course pattern

Before doing any analysis, always inspect: `.describe()`, `.head()`, `.tail()`, check for NaNs.

In [ ]:
print('=== describe() ===')
print(df_distraction.describe().round(2))

print('\n=== Any NaN values? ===')
print(df_distraction.isnull().sum())

### 3b. Key statistics — what does the OECD average actually mean?

In [ ]:
# Separate the OECD average row from country-level data
oecd_row = df_distraction[df_distraction['country'] == 'OECD average']
countries = df_distraction[df_distraction['country'] != 'OECD average'].copy()

print('OECD average row:')
print(oecd_row.to_string(index=False))

print(f'\nNumber of countries/economies: {len(countries)}')
print(f'Mean self-distraction across all: {countries["self_distracted_pct"].mean():.2f}%')
print(f'Min: {countries["self_distracted_pct"].min():.2f}% ({countries.loc[countries["self_distracted_pct"].idxmin(), "country"]})')
print(f'Max: {countries["self_distracted_pct"].max():.2f}% ({countries.loc[countries["self_distracted_pct"].idxmax(), "country"]})')

### 3c. Binning countries by distraction level

Using `pd.cut()` to group countries into distraction bands — directly mirroring the binning exercise in the pandas course (Section 6 of the CM Hub course).

In [ ]:
bins   = [0, 20, 30, 40, 55]
labels = ['Low (<20%)', 'Medium (20–30%)', 'High (30–40%)', 'Very high (40%+)']

countries['distraction_band'] = pd.cut(
    countries['self_distracted_pct'],
    bins=bins,
    labels=labels
)

band_counts = countries['distraction_band'].value_counts().sort_index()
print('Countries per distraction band:')
print(band_counts)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Bar chart: countries per band
colours = ['#3ECFB2', '#F5C842', '#E8472A', '#8B3018']
band_counts.plot(kind='bar', ax=axes[0], color=colours, edgecolor='#111', linewidth=1.2, rot=20)
axes[0].set_title('Countries by distraction band', fontweight='bold', pad=12)
axes[0].set_xlabel('')
axes[0].set_ylabel('Number of countries')
axes[0].axhline(y=band_counts.mean(), color='#111', linestyle='--', linewidth=1, alpha=0.4, label='Mean')
axes[0].legend()

# Scatter: self vs others distraction
axes[1].scatter(
    countries['self_distracted_pct'],
    countries['others_distracted_pct'],
    alpha=0.65, color='#E8472A', edgecolors='#111', linewidth=0.8, s=60
)
# Highlight UK
uk = countries[countries['country'] == 'United Kingdom']
axes[1].scatter(uk['self_distracted_pct'], uk['others_distracted_pct'],
                color='#3B82F6', s=120, zorder=5, edgecolors='#111', linewidth=1.2)
axes[1].annotate('UK', xy=(uk['self_distracted_pct'].values[0], uk['others_distracted_pct'].values[0]),
                 xytext=(20, 22), fontsize=9, fontweight='bold', color='#3B82F6')
# OECD average lines
axes[1].axvline(x=30.45, color='#111', linestyle='--', linewidth=1, alpha=0.4)
axes[1].axhline(y=25.22, color='#111', linestyle='--', linewidth=1, alpha=0.4)
axes[1].set_title('Self-distracted vs distracted by others', fontweight='bold', pad=12)
axes[1].set_xlabel('Self-distracted (%)')
axes[1].set_ylabel('Distracted by others (%)')

plt.tight_layout()
plt.savefig('fig1_distraction_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig1_distraction_analysis.png')

### 3d. Correlation between self-distraction and others-distraction

Are countries where students distract themselves also the ones where classmates distract each other more? Let's check with a correlation matrix.

In [ ]:
print('Correlation matrix:')
print(countries[['self_distracted_pct', 'others_distracted_pct']].corr().round(3))
print('\nA strong positive correlation (r ≈ 0.9) would suggest these two behaviours go hand in hand.')

### 3e. Sort and filter — top and bottom 10 countries

In [ ]:
# Sort ascending
sorted_countries = countries.sort_values('self_distracted_pct')

print('=== 10 LEAST distracted countries ===')
print(sorted_countries.head(10)[['country', 'self_distracted_pct']].to_string(index=False))

print('\n=== 10 MOST distracted countries ===')
print(sorted_countries.tail(10)[['country', 'self_distracted_pct']].to_string(index=False))

print('\n=== UK specifically ===')
print(countries[countries['country'].str.contains('Kingdom')][['country', 'self_distracted_pct', 'others_distracted_pct']].to_string(index=False))

---
## 4. Load & clean Figure 3 — Screen time vs maths scores

**What this sheet contains:** OECD average maths scores for students grouped by how many hours per day they spend on each leisure activity (gaming, social media, browsing, etc.).

This is the data that powers the D3 scrollytelling chart in the website.

In [ ]:
# The scores data is embedded in the sheet - we build it from known OECD averages
# (extracted via the extract-text tool during the project's data exploration phase)

time_labels = ['None', 'Up to 1h', '1–3h', '3–5h', '5–7h', '7h+']

scores_data = {
    'time_bracket': time_labels,
    'learning_at_school': [456.14, 480.70, 476.17, 479.35, 477.33, 465.32],
    'leisure_at_school':  [471.83, 491.60, 482.95, 468.46, 451.28, 430.04],
    'gaming':             [480.40, 495.69, 494.24, 475.29, 459.59, 440.83],
    'social_media':       [477.12, 500.80, 499.90, 472.89, 450.48, 428.95],
    'browsing_internet':  [463.19, 501.49, 494.61, 470.58, 453.46, 438.51]
}

df_scores = pd.DataFrame(scores_data)
df_scores = df_scores.set_index('time_bracket')
print('Scores DataFrame:')
print(df_scores.round(1))

### 4a. Describe the scores data

In [ ]:
print('=== describe() — scores by activity type ===')
print(df_scores.describe().round(2))

print('\n=== Which time bracket has the highest mean score across all activities? ===')
print(df_scores.mean(axis=1).round(2).sort_values(ascending=False))

### 4b. Tidy data — reshape from wide to long format

The DataFrame above is in **wide format** — each activity is a column. For plotting and grouping it's more useful in **long (tidy) format** — one row per observation.

This directly applies the tidy data principles from Wickham (2014), referenced in the course materials: *"Each variable forms a column, each observation forms a row."*

In [ ]:
# Reset index to make time_bracket a regular column before melting
df_scores_tidy = df_scores.reset_index().melt(
    id_vars='time_bracket',
    var_name='activity',
    value_name='mean_maths_score'
)

# Make activity names readable
activity_labels = {
    'learning_at_school': 'Learning (at school)',
    'leisure_at_school':  'Leisure (at school)',
    'gaming':             'Gaming',
    'social_media':       'Social media',
    'browsing_internet':  'Browsing internet'
}
df_scores_tidy['activity'] = df_scores_tidy['activity'].map(activity_labels)

print('Tidy format (first 12 rows):')
print(df_scores_tidy.head(12).to_string(index=False))
print(f'\nTotal rows: {len(df_scores_tidy)} ({len(df_scores)} time brackets × {len(df_scores.columns)} activities)')

### 4c. Group by activity — find the peak and drop for each

In [ ]:
# Group by activity, find max and min score
grouped = df_scores_tidy.groupby('activity')['mean_maths_score']

summary = pd.DataFrame({
    'peak_score': grouped.max(),
    'lowest_score': grouped.min(),
    'score_drop': grouped.max() - grouped.min()
}).round(2)

summary = summary.sort_values('score_drop', ascending=False)
print('Score drop from peak to lowest, by activity type:')
print(summary.to_string())
print('\nThe activity with the biggest drop is the most harmful for heavy users.')

### 4d. Plot — scores by screen time (the sweet spot visualisation)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))

colours_map = {
    'Learning (at school)': '#3B82F6',
    'Gaming':               '#E8472A',
    'Social media':         '#c8860a',
    'Browsing internet':    '#3ECFB2',
    'Leisure (at school)':  '#C4A8F0'
}

for activity, grp in df_scores_tidy.groupby('activity'):
    ax.plot(
        grp['time_bracket'],
        grp['mean_maths_score'],
        marker='o', linewidth=2.5, markersize=7,
        color=colours_map.get(activity, '#888'),
        label=activity
    )

# Sweet spot shading
ax.axvspan(0.8, 1.2, alpha=0.08, color='#3ECFB2', label='_nolegend_')
ax.axvline(x=1, color='#3ECFB2', linestyle='--', linewidth=1, alpha=0.5)
ax.text(1.05, 502, '🍭 sweet spot', fontsize=9, color='#2aaa90', fontweight='bold')

ax.set_title('PISA 2022: Maths scores by daily screen time (OECD average)',
             fontweight='bold', fontsize=13, pad=14)
ax.set_xlabel('Daily screen time bracket', fontsize=11)
ax.set_ylabel('Mean maths score (OECD average)', fontsize=11)
ax.legend(loc='lower left', fontsize=9)
ax.set_ylim(420, 510)
ax.yaxis.set_major_locator(mticker.MultipleLocator(10))

plt.tight_layout()
plt.savefig('fig3_scores_by_screentime.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig3_scores_by_screentime.png')

---
## 5. Load & clean Table 44 — Sense of belonging

**What this sheet contains:** OECD average 'sense of belonging' index scores for students grouped by leisure screen time. Used in the belonging mood-meter section of the website.

The belonging index is a standardised score — positive means more connected, negative means more isolated.

In [ ]:
# Sense of belonging data — OECD averages from Tables 43 (learning) and 44 (leisure)
belonging_data = {
    'time_bracket': ['None', 'Up to 1h', '1–2h', '2–3h', '3–5h', '5–7h', '7h+'],
    'leisure_belonging':  [-0.0176,  0.0183, -0.0066, -0.0346, -0.0656, -0.0836, -0.0720],
    'learning_belonging': [-0.0837, -0.0086,  0.0071,  0.0061,  0.0042, -0.0161, -0.0071]
}

df_belonging = pd.DataFrame(belonging_data)
df_belonging = df_belonging.set_index('time_bracket')

print('Belonging index by screen time:')
print(df_belonging.round(4))

print('\nNote: negative = lower sense of belonging (more isolated)')
print('Leisure screen time drops belonging more sharply than learning screen time.')

In [ ]:
print('=== describe() — belonging indices ===')
print(df_belonging.describe().round(4))

print('\n=== Lowest belonging score (leisure) ===')
min_idx = df_belonging['leisure_belonging'].idxmin()
print(f'{min_idx}: {df_belonging.loc[min_idx, "leisure_belonging"]:.4f}')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

x = range(len(df_belonging))
ax.plot(x, df_belonging['leisure_belonging'],  marker='o', color='#E8472A',
        linewidth=2.5, markersize=8, label='Leisure screen time')
ax.plot(x, df_belonging['learning_belonging'], marker='s', color='#3B82F6',
        linewidth=2.5, markersize=8, label='Learning screen time', linestyle='--')

ax.axhline(0, color='#111', linewidth=1, alpha=0.3, linestyle=':')
ax.fill_between(x, df_belonging['leisure_belonging'], 0,
                where=[v < 0 for v in df_belonging['leisure_belonging']],
                alpha=0.1, color='#E8472A', label='_nolegend_')

ax.set_xticks(list(x))
ax.set_xticklabels(df_belonging.index, fontsize=10)
ax.set_title('PISA 2022: Sense of belonging by screen time type (OECD average)',
             fontweight='bold', fontsize=12, pad=12)
ax.set_xlabel('Daily screen time bracket')
ax.set_ylabel('Belonging index (0 = neutral)')
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig('fig_belonging.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig_belonging.png')

---
## 6. Export to JSON — feeding the D3 visualisation

The final step links Python data analysis to the browser visualisation. We export cleaned data to JSON so the D3.js chart can load it. This is the bridge between the Python pipeline and the JavaScript front-end.

This also demonstrates the full tidy data principle in practice: raw Excel → cleaned pandas DataFrame → tidy long format → structured JSON for visualisation.

In [ ]:
# Build JSON output for the D3 chart
output = {
    'source': 'OECD PISA 2022 – PIF N°124',
    'doi': 'https://doi.org/10.1787/53f23881-en',
    'time_labels': time_labels,
    'scores': {
        'learning': [round(v, 1) for v in df_scores['learning_at_school'].values.tolist()],
        'gaming':   [round(v, 1) for v in df_scores['gaming'].values.tolist()],
        'social':   [round(v, 1) for v in df_scores['social_media'].values.tolist()],
        'browse':   [round(v, 1) for v in df_scores['browsing_internet'].values.tolist()]
    },
    'belonging': {
        'leisure':  [round(v, 4) for v in df_belonging['leisure_belonging'].values.tolist()],
        'learning': [round(v, 4) for v in df_belonging['learning_belonging'].values.tolist()]
    },
    'distraction_by_country': [
        {
            'country': row['country'],
            'self_pct': row['self_distracted_pct'],
            'others_pct': row['others_distracted_pct']
        }
        for _, row in sorted_countries.iterrows()
    ]
}

with open('pisa_data.json', 'w') as f:
    json.dump(output, f, indent=2)

print('Exported: pisa_data.json')
print(f'Countries exported: {len(output["distraction_by_country"])}')
print(f'Score data points: {len(output["scores"]["learning"])} time brackets × 4 activities')
print(f'Belonging data points: {len(output["belonging"]["leisure"])} time brackets × 2 types')

---
## 7. Summary — what this analysis found

This notebook documents the full Python data pipeline for the project.

In [ ]:
print('=== PROJECT DATA SUMMARY ===')
print(f'Dataset: OECD PISA 2022 – PIF N°124 (Excel, {len(xl.sheet_names)} sheets)')
print()
print(f'Countries analysed: {len(countries)}')
print(f'OECD avg self-distraction in class: {oecd_row["self_distracted_pct"].values[0]:.1f}%')
print(f'OECD avg distracted by others: {oecd_row["others_distracted_pct"].values[0]:.1f}%')
print()
print('Score drop from 1hr to 7hr+ leisure (OECD avg):')
for act in ['social_media', 'gaming', 'browsing_internet', 'learning_at_school']:
    peak = df_scores.loc['Up to 1h', act]
    low  = df_scores.loc['7h+', act]
    print(f'  {act:25s}: {peak:.0f} → {low:.0f}  (drop: {peak-low:.0f} pts)')
print()
print('Belonging index drop (leisure, 1hr → 5-7hr):')
print(f'  {df_belonging.loc["Up to 1h", "leisure_belonging"]:.4f} → {df_belonging.loc["5–7h", "leisure_belonging"]:.4f}')
print()
print('Files saved:')
print('  pisa_data.json              — data for D3 visualisation')
print('  fig1_distraction_analysis.png  — country distraction charts')
print('  fig3_scores_by_screentime.png  — scores by screen time')
print('  fig_belonging.png              — belonging index chart')

---
## References

- OECD (2023). *Students, Computers and Learning: Making the Connection*. PIF N°124 – PISA 2022 Results Volume II. https://doi.org/10.1787/53f23881-en
- McKinney, W. (2022). *Python for Data Analysis*, 3rd ed. O'Reilly Media. https://wesmckinney.com/book/
- Wickham, H. (2014). Tidy Data. *Journal of Statistical Software*, 59(10), 1–23. https://r4ds.had.co.nz/tidy-data.html
- CM Hub, Imperial College (2022). *Data Processing with Pandas*. https://github.com/Pecnut/course-pandas
- OECD (2024). *Screen Time and Subjective Well-being*. https://www.oecd.org/en/publications/screen-time-and-subjective-well-being_c840403c-en/full-report.html